### Part1- Load and inspect dataset.

In [7]:
import pandas as pd 

In [8]:
path = "data/penguins_with_splits.csv"
data = pd.read_csv(path)
rows, columns = data.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {columns}")
data.head()

Number of rows: 344
Number of columns: 9


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,split
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007,train
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007,val
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007,train
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007,train
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007,train


In [9]:
print(data["split"].value_counts())
print(data.isnull().sum())

split
train    241
val       69
test      34
Name: count, dtype: int64
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
split                 0
dtype: int64


### Part2- Handling Missing Values

In [10]:
study = data.copy()

numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

study = study.dropna(subset=numeric_cols).copy()
print(f"study shape: {study.shape}")


study["bill_shape_ratio"] = (
  study["bill_length_mm"] / study["bill_depth_mm"]
)

y = study["species"].copy()

x = study[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "island", "bill_shape_ratio"]].copy() # scaled_features

print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")
x.head()

study shape: (342, 9)
x shape: (342, 6)
y shape: (342,)


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island,bill_shape_ratio
0,39.1,18.7,181.0,3750.0,Torgersen,2.090909
1,39.5,17.4,186.0,3800.0,Torgersen,2.270115
2,40.3,18.0,195.0,3250.0,Torgersen,2.238889
4,36.7,19.3,193.0,3450.0,Torgersen,1.901554
5,39.3,20.6,190.0,3650.0,Torgersen,1.907767


### Part 3 — Transform Feature 

In [11]:
train = study[study["split"] == "train"].copy()

numeric_features = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "bill_shape_ratio",
]

mean = train[numeric_features].mean()
std = train[numeric_features].std()

scaled_features = (study[numeric_features] - mean) / std

print("Scaled Features Shape:", scaled_features.shape)
print(scaled_features.head())


train_categories = train["island"].unique()

print("Training Categories:", train_categories)

island_encoded = pd.DataFrame(index=study.index)

for category in train_categories:
    island_encoded[f"island_{category}"] = (study["island"] == category).astype(int)

print("Encoded Features Shape:", island_encoded.shape)
print(island_encoded.head())

Scaled Features Shape: (342, 5)
   bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  \
0       -0.904286       0.757836          -1.413822    -0.564158   
1       -0.831767       0.082199          -1.056705    -0.502814   
2       -0.686730       0.394031          -0.413896    -1.177597   
4       -1.339400       1.069669          -0.556743    -0.932221   
5       -0.868027       1.745307          -0.771012    -0.686846   

   bill_shape_ratio  
0         -1.024781  
1         -0.664344  
2         -0.727149  
4         -1.405631  
5         -1.393135  
Training Categories: ['Torgersen' 'Biscoe' 'Dream']
Encoded Features Shape: (342, 3)
   island_Torgersen  island_Biscoe  island_Dream
0                 1              0             0
1                 1              0             0
2                 1              0             0
4                 1              0             0
5                 1              0             0


### Part 4 - incpect final data 

In [12]:
final_features = pd.concat([scaled_features, island_encoded], axis=1)

print("Final Features:")
print(final_features.head())

print("\nFinal Shape:", final_features.shape)

print("\nData Types:")
print(final_features.dtypes)

print("\nMissing Values:")
print(final_features.isnull().sum())

print("Total Missing Values:", final_features.isnull().sum().sum())

print("\nSpecies Excluded:", "species" not in final_features.columns)
print("Split Excluded:", "split" not in final_features.columns)

print("\nAll Numeric:",
      all(pd.api.types.is_numeric_dtype(dtype)
          for dtype in final_features.dtypes))

print("\nOriginal Features:")
print(x.head())

print("\nTransformed Features:")
print(final_features.head())

Final Features:
   bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  \
0       -0.904286       0.757836          -1.413822    -0.564158   
1       -0.831767       0.082199          -1.056705    -0.502814   
2       -0.686730       0.394031          -0.413896    -1.177597   
4       -1.339400       1.069669          -0.556743    -0.932221   
5       -0.868027       1.745307          -0.771012    -0.686846   

   bill_shape_ratio  island_Torgersen  island_Biscoe  island_Dream  
0         -1.024781                 1              0             0  
1         -0.664344                 1              0             0  
2         -0.727149                 1              0             0  
4         -1.405631                 1              0             0  
5         -1.393135                 1              0             0  

Final Shape: (342, 8)

Data Types:
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
bill_shap